# Computational experiment template

This notebook creates a reproducible benchmark table for the paper. Start with the small grid below, verify licenses and output, and then increase sizes and time limits.

In [ ]:
from pathlib import Path
import numpy as np
import sys
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd
from sdp_clusters import Data, WarmStartOptions, build_warm_start, solve_compact, solve_pairwise, solve_mpco

In [2]:
SIZES = [10, 15, 20]
FACILITIES = [2, 3]
TAUS = [1, 1.5, 2, 3, np.inf]
SEEDS = [0, 1, 2]
TIME_LIMIT = 60
METHODS = {
    'compact': solve_compact,
    'pairwise': solve_pairwise,
    'mpco': solve_mpco,
}

NameError: name 'np' is not defined

In [ ]:
records = []
for n in SIZES:
    for p in FACILITIES:
        if p > n:
            continue
        for tau in TAUS:
            for seed in SEEDS:
                data = Data.blobs(n=n, facilities=p, spread=0.8, seed=seed, tau=tau)
                warm = build_warm_start(data, WarmStartOptions(
                    strategy='best', random_starts=5, alternating_iterations=30,
                    sdp_time_limit=min(20, TIME_LIMIT), seed=seed))
                for method_name, solver in METHODS.items():
                    result = solver(data, time_limit=TIME_LIMIT, warm_start=warm)
                    record = result.summary()
                    record.update(
                        instance=data.name, n=n, p=p, tau=tau, seed=seed,
                        sdp_root_bound=result.metadata.get('sdp_root_bound', np.nan),
                        sdp_bound_time=result.metadata.get('sdp_bound_time', np.nan),
                        sdp_root_gap=result.metadata.get('sdp_root_gap', np.nan),
                        continuous_relaxation_bound=result.metadata.get('continuous_relaxation_bound', np.nan),
                        continuous_relaxation_gap=result.metadata.get('continuous_relaxation_gap', np.nan),
                        continuous_relaxation_time=result.metadata.get('continuous_relaxation_time', np.nan),
                        projection_logic_cuts=result.metadata.get('projection_logic_cuts', 0),
                    )
                    records.append(record)
                    print(data.name, f'p={p}', f'tau={tau}', method_name,
                          result.status, result.objective, result.gap)
table = pd.DataFrame(records)
table

In [ ]:
Path('results').mkdir(exist_ok=True)
table.to_csv('results/benchmark.csv', index=False)
summary = table.groupby(['n', 'p', 'tau', 'method']).agg(
    mean_time=('runtime', 'mean'), median_gap=('gap', 'median'),
    median_relaxation_gap=('continuous_relaxation_gap', 'median'),
    median_sdp_gap=('sdp_root_gap', 'median'),
    solved=('optimal', 'sum'), mean_objective=('objective', 'mean')
).reset_index()
summary

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
for method, group in summary.groupby('method'):
    by_n = group.groupby('n').mean(numeric_only=True)
    axes[0].plot(by_n.index, by_n['mean_time'], marker='o', label=method)
    axes[1].plot(by_n.index, by_n['median_gap'], marker='o', label=method)
    axes[2].plot(by_n.index, by_n['median_relaxation_gap'], marker='^', label=method)
axes[0].set(xlabel='n', ylabel='Mean runtime (s)', title='Computational performance')
axes[1].set(xlabel='n', ylabel='Median relative gap', title='Proof quality')
axes[2].set(xlabel='n', ylabel='Median initial relaxation gap', title='Relaxation quality')
for ax in axes:
    ax.legend()
    ax.grid(alpha=0.25)
fig.tight_layout()
plt.show()

For the paper, retain raw per-instance records. Aggregate tables alone hide time-limit censoring and variation across seeds. Report objectives, lower bounds, relative gaps, times, statuses, and decomposition iterations.